# Praktikum 2 — Strong Baseline: Perbandingan Klasik yang Adil

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/GIGA-TDDM/giga-tddm-pjj-2026-praktikum-baseline/blob/main/notebooks/praktikum02_baseline.ipynb)

**EF235161 Topik Dalam Data Mining (P) — S-2 Teknik Informatika, ITS**
Classification, Regression & Strong Baselines · pengerjaan asinkron, ±3 jam

---

## Isi identitas Anda

| | |
|---|---|
| **Nama** |Alisha Anggranidi Salsabila|
| **NRP** | 6025261014 |
| **Username GitHub** | alishaanggranidi |

---

## Pesan utama

> **Klaim "metode saya lebih baik" hanya sekuat baseline-nya.**
> Praktikum ini melatih Anda membangun baseline yang layak dipercaya: tuning dengan anggaran setara, evaluasi yang tidak bocor, hasil dilaporkan dengan sebaran, dan perbedaan diuji sebelum diklaim.

## Cara menyimpan pekerjaan Anda

Bekerja di Colab, **sekali di awal**:

1. Klik **Copy to Drive** di bar atas. Sebelum ini, pilihan simpan ke GitHub belum muncul.
2. `File → Open notebook → tab GitHub`, centang **Include private repositories**, lalu *Authorize*.

Setiap kali ingin menyimpan: `File → Save a copy in GitHub` → repositori praktikum Anda → branch `main` → **pastikan path-nya** `notebooks/praktikum02_baseline.ipynb`. Itulah commit Anda.

Bila tetap gagal: `File → Download → Download .ipynb`, lalu unggah lewat `Add file → Upload files` di repositori Anda (folder `notebooks/`, nama sama).

## Aturan

- Jalankan sel **berurutan dari atas**. Jangan melompat.
- Setelah setiap blok, isi satu sel **Interpretasi**. Menjalankan kode tanpa interpretasi berarti belum mengerjakan praktikum.
- **Test set dibuka satu kali** (Blok 9). Tidak ada sel lain yang boleh menyentuhnya.
- **Anggaran tuning sama untuk semua model** (`TUNING_BUDGET`). Jangan memberi model favorit Anda lebih banyak percobaan.
- Tugas kode Anda ada di **Blok 5**: menambah satu model dari keluarga berbeda.
- Runtime penuh ±3–5 menit di Colab. Sabar pada Blok 5.

## Blok 0 — Setup & reproducibility

In [1]:
import sys, os, time, warnings
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import scipy, sklearn

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 160)
os.makedirs("outputs", exist_ok=True)
T_MULAI = time.time()

print("Python      :", sys.version.split()[0])
print("NumPy       :", np.__version__)
print("pandas      :", pd.__version__)
print("SciPy       :", scipy.__version__)
print("scikit-learn:", sklearn.__version__)
print("RANDOM_STATE:", RANDOM_STATE)

Python      : 3.13.15
NumPy       : 2.1.3
pandas      : 2.2.3
SciPy       : 1.16.3
scikit-learn: 1.6.1
RANDOM_STATE: 42


Versi library dan seed dicatat sebelum analisis apa pun. Hasil Anda boleh berbeda tipis dari teman Anda bila versi scikit-learn berbeda; itu alasan log ini ada.

**Interpretasi** (1–2 kalimat, sebut angkanya): Notebook menggunakan Python 3.13.15, scikit-learn 1.6.1, SciPy 1.16.3, RANDOM_STATE=42.

## Blok 1 — Dua dataset, dua tingkat kesulitan

Baseline yang "kuat" bukan sifat model, melainkan sifat **pasangan model dan data**. Karena itu kita memakai dua dataset:

- **A — Breast Cancer Wisconsin** (bawaan scikit-learn): dataset yang dikenal **mudah**. Kelas positif sengaja kita tetapkan = *malignant*, karena itu yang ingin ditangkap.
- **B — Sintetis** (`make_classification`, dibangkitkan di sini): tidak seimbang (±10% positif), ada noise label dan struktur non-linear. Ini dataset **sulit** buatan, hanya untuk latihan.

Data sintetis tidak boleh dipakai sebagai dasar klaim empiris. Lihat `data/DATA_CARD.md`.

In [2]:
from sklearn.datasets import load_breast_cancer, make_classification

_bc = load_breast_cancer(as_frame=True)
X_A = _bc.data
y_A = (_bc.target == 0).astype(int)          # 1 = malignant (yang ingin ditangkap)

_Xb, _yb = make_classification(
    n_samples=1500, n_features=20, n_informative=6, n_redundant=2,
    n_clusters_per_class=3, weights=[0.9, 0.1], flip_y=0.03,
    class_sep=0.8, random_state=RANDOM_STATE)
X_B = pd.DataFrame(_Xb, columns=[f"f{i:02d}" for i in range(20)])
y_B = pd.Series(_yb)

DATASET = {
    "A (mudah)": (X_A, y_A),
    "B (sulit)": (X_B, y_B),
}
for nama, (X, y) in DATASET.items():
    print(f"{nama:10s} ukuran={X.shape}  positif={100*y.mean():5.1f}%  "
          f"akurasi tebakan-mayoritas={max(y.mean(), 1-y.mean()):.3f}")

A (mudah)  ukuran=(569, 30)  positif= 37.3%  akurasi tebakan-mayoritas=0.627
B (sulit)  ukuran=(1500, 20)  positif= 10.8%  akurasi tebakan-mayoritas=0.892


Perhatikan angka **akurasi tebakan-mayoritas**. Nilai itu adalah lantai yang harus dilampaui setiap model.

**Interpretasi** (1–2 kalimat, sebut angkanya): Dataset A berisi 569 baris dengan akurasi 37,3 % positif tapi tebakannya mayoritas 0,627. Dataset B berisi 15000 baris dengan 10,8% positif tapi tebakannya mayoritas 0,892.

## Blok 2 — Protokol yang adil (dibaca dulu, sebelum menjalankan)

Empat aturan yang berlaku untuk **semua** model, tanpa pengecualian:

1. **Split sekali, di awal, stratified.** Test set dikunci sampai Blok 9.
2. **Semua preprocessing di dalam `Pipeline`.** Scaler belajar hanya dari fold latih.
3. **Anggaran tuning setara.** Setiap model mendapat `TUNING_BUDGET` percobaan yang sama pada folds yang sama.
4. **Estimasi tuning yang jujur: nested cross-validation.** Tuning di lipatan dalam, penilaian di lipatan luar. Memilih hyperparameter dan melaporkan skor dari folds yang sama membuat hasil optimis (*model selection bias*).

In [3]:
from sklearn.model_selection import (train_test_split, RepeatedStratifiedKFold,
                                     StratifiedKFold, cross_validate, RandomizedSearchCV)

TUNING_BUDGET = 12                       # sama untuk SEMUA model
CV_LUAR  = RepeatedStratifiedKFold(n_splits=5, n_repeats=2, random_state=RANDOM_STATE)
CV_DALAM = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)
SCORING  = {"roc_auc": "roc_auc", "pr_auc": "average_precision",
            "f1": "f1", "accuracy": "accuracy"}
METRIK_UTAMA = "pr_auc"                  # dipilih karena data tidak seimbang; alasannya Anda tulis di laporan

SPLIT = {}
for nama, (X, y) in DATASET.items():
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE)
    SPLIT[nama] = (X_train, X_test, y_train, y_test)
    print(f"{nama}: train={X_train.shape}  test={X_test.shape}  "
          f"positif train={y_train.mean():.3f}  test={y_test.mean():.3f}")
print("\nMulai detik ini test set dikunci. Dibuka satu kali di Blok 9.")

A (mudah): train=(455, 30)  test=(114, 30)  positif train=0.374  test=0.368
B (sulit): train=(1200, 20)  test=(300, 20)  positif train=0.108  test=0.107

Mulai detik ini test set dikunci. Dibuka satu kali di Blok 9.


Mengapa `METRIK_UTAMA = "pr_auc"`, bukan accuracy? Tulis satu kalimat.

**Interpretasi** (1–2 kalimat, sebut angkanya): Proporsi positif di train vs test hampir sama persis karena adanya stratify (A: 37,4% vs 36,8%, B: 10,8% vs 10,7%). pr_auc digunakan karena Dataset B timpang, jadi akurasinya tinggi, bisa jadi cuman dari nebak mayoritas, bukan beneran ngenalin kasus positif.

## Blok 3 — Baseline yang tidak berpikir

Sebelum model apa pun, ukur model yang hanya menebak kelas mayoritas. Semua model lain dinilai terhadap garis ini.

In [4]:
from sklearn.dummy import DummyClassifier

def ringkas(hasil_cv):
    """dict metrik -> (rata-rata, std) dari hasil cross_validate."""
    return {m: (hasil_cv["test_" + m].mean(), hasil_cv["test_" + m].std()) for m in SCORING}

def tampilkan(tabel, dataset):
    baris = []
    for (d, model, kondisi), r in tabel.items():
        if d != dataset:
            continue
        baris.append({"model": model, "kondisi": kondisi,
                      **{m: f"{r[m][0]:.3f} ± {r[m][1]:.3f}" for m in SCORING}})
    return pd.DataFrame(baris)

TABEL = {}          # (dataset, model, kondisi) -> ringkasan metrik
for nama in DATASET:
    X_train, _, y_train, _ = SPLIT[nama]
    dummy = DummyClassifier(strategy="most_frequent")
    TABEL[(nama, "Dummy (mayoritas)", "-")] = ringkas(
        cross_validate(dummy, X_train, y_train, cv=CV_LUAR, scoring=SCORING))

for nama in DATASET:
    print(f"\n=== Dataset {nama} ===")
    print(tampilkan(TABEL, nama).to_string(index=False))


=== Dataset A (mudah) ===
            model kondisi       roc_auc        pr_auc            f1      accuracy
Dummy (mayoritas)       - 0.500 ± 0.000 0.374 ± 0.000 0.000 ± 0.000 0.626 ± 0.000

=== Dataset B (sulit) ===
            model kondisi       roc_auc        pr_auc            f1      accuracy
Dummy (mayoritas)       - 0.500 ± 0.000 0.108 ± 0.000 0.000 ± 0.000 0.892 ± 0.000


Baca **accuracy** dan **pr_auc** Dummy pada kedua dataset. Apa yang dikatakan angka itu tentang metrik mana yang layak?

**Interpretasi** (1–2 kalimat, sebut angkanya): Dummy dapet accuracy 0,626 (A) dan 0,892 (B) padahal cuman nebak mayoritas . acc nya kelihatan bagus. namun pada roc_auc (0,500) dan f1 (0,000) di kedua dataset justru jujur nunjukkin model ini gaada belajar.

## Blok 4 — Tiga model klasik dengan pengaturan bawaan

Inilah keadaan yang sering muncul di paper: baseline dijalankan **tanpa tuning**. Catat hasilnya; di Blok 5 kita lihat apa yang berubah setelah tuning yang adil.

In [5]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier

def buat_model_wajib():
    return {
        "Logistic Regression": Pipeline([
            ("scaler", StandardScaler()),
            ("clf", LogisticRegression(max_iter=2000, random_state=RANDOM_STATE))]),
        "SVM (RBF)": Pipeline([
            ("scaler", StandardScaler()),
            ("clf", SVC(probability=True, random_state=RANDOM_STATE))]),
        "Random Forest": Pipeline([
            ("clf", RandomForestClassifier(random_state=RANDOM_STATE))]),
    }
#menggunakan standardscaler karena lr dan svm sensitif ke skala fitur

for nama in DATASET:
    X_train, _, y_train, _ = SPLIT[nama]
    for model, est in buat_model_wajib().items():
        TABEL[(nama, model, "default")] = ringkas(
            cross_validate(est, X_train, y_train, cv=CV_LUAR, scoring=SCORING))

for nama in DATASET:
    print(f"\n=== Dataset {nama} (CV luar 5x2, tanpa tuning) ===")
    print(tampilkan(TABEL, nama).to_string(index=False))


=== Dataset A (mudah) (CV luar 5x2, tanpa tuning) ===
              model kondisi       roc_auc        pr_auc            f1      accuracy
  Dummy (mayoritas)       - 0.500 ± 0.000 0.374 ± 0.000 0.000 ± 0.000 0.626 ± 0.000
Logistic Regression default 0.996 ± 0.005 0.995 ± 0.005 0.967 ± 0.021 0.976 ± 0.015
          SVM (RBF) default 0.995 ± 0.005 0.993 ± 0.006 0.961 ± 0.021 0.971 ± 0.015
      Random Forest default 0.989 ± 0.009 0.987 ± 0.010 0.943 ± 0.039 0.958 ± 0.028

=== Dataset B (sulit) (CV luar 5x2, tanpa tuning) ===
              model kondisi       roc_auc        pr_auc            f1      accuracy
  Dummy (mayoritas)       - 0.500 ± 0.000 0.108 ± 0.000 0.000 ± 0.000 0.892 ± 0.000
Logistic Regression default 0.764 ± 0.052 0.454 ± 0.055 0.321 ± 0.058 0.906 ± 0.004
          SVM (RBF) default 0.811 ± 0.068 0.561 ± 0.066 0.226 ± 0.069 0.905 ± 0.004
      Random Forest default 0.797 ± 0.056 0.536 ± 0.065 0.274 ± 0.081 0.908 ± 0.007


In [11]:
def summary_model(tabel, dataset, kondisi=None):
    baris = []
    for (d, model, kond), r in tabel.items():
        if d != dataset or (kondisi and kond != kondisi):
            continue
        baris.append({"model": model, "kondisi": kond,
                      **{m: r[m][0] for m in SCORING}})
    return pd.DataFrame(baris).sort_values("pr_auc", ascending=False)

for nama in DATASET:
    df_show = summary_model(TABEL, nama)
    display(
        df_show.style.set_caption(f"Dataset {nama} — Dummy vs default, urut pr_auc")
        .background_gradient(subset=list(SCORING), cmap="RdPu")
        .format("{:.3f}", subset=list(SCORING))
    )

,model,kondisi,roc_auc,pr_auc,f1,accuracy
1,Logistic Regression,default,0.996,0.995,0.967,0.976
2,SVM (RBF),default,0.995,0.993,0.961,0.971
3,Random Forest,default,0.989,0.987,0.943,0.958
0,Dummy (mayoritas),-,0.500,0.374,0.000,0.626


,model,kondisi,roc_auc,pr_auc,f1,accuracy
2,SVM (RBF),default,0.811,0.561,0.226,0.905
3,Random Forest,default,0.797,0.536,0.274,0.908
1,Logistic Regression,default,0.764,0.454,0.321,0.906
0,Dummy (mayoritas),-,0.500,0.108,0.000,0.892


**Interpretasi** (1–2 kalimat, sebut angkanya): Di Dataset A, ketiga model udah deket ceiling (pr_auc 0,987–0,995) dengan std kecil (≤0,010), jadi hasilnya stabil dan semua model emang bagus. Di Dataset B, pr_auc jauh lebih rendah dan gap antar model (0,454–0,561) itu masih kepotong sama std nya sendiri (0,055–0,066), jadi belum bisa langsung klaim SVM (0,561) lebih baik dari Random Forest (0,536).

## Blok 5 — Tuning dengan anggaran setara (nested CV) dan **tugas kode Anda**

### Tugas: tambahkan satu model dari keluarga berbeda

Isi `MODEL_SAYA` di sel berikut dengan **satu atau lebih model tambahan** (k-NN, Decision Tree, Naive Bayes, atau lainnya yang Anda pilih dan bisa pertanggungjawabkan). Setiap entri berisi `(Pipeline, distribusi_hyperparameter)`. Ingat: model yang butuh scaling (k-NN, SVM) harus memasukkan scaler **di dalam** Pipeline.

Anda tidak perlu mengatur anggaran tuning: fungsi `evaluasi_nested` memakai `TUNING_BUDGET` yang sama untuk semua model. Itulah inti keadilannya.

Contoh bentuk (buka komentar, ubah, atau tulis milik Anda sendiri):

In [12]:
from scipy import stats
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB

# --- Isi di sini. Hapus tanda # pada contoh lalu sesuaikan, atau tulis modelmu sendiri. ---
# MODEL_SAYA = {
#     "k-NN": (
#         Pipeline([("scaler", StandardScaler()), ("clf", KNeighborsClassifier())]),
#         {"clf__n_neighbors": [3, 5, 7, 11, 15, 21, 31], "clf__weights": ["uniform", "distance"]},
#     ),
# }
MODEL_SAYA = {
    "k-NN": (
        Pipeline([("scaler", StandardScaler()), ("clf", KNeighborsClassifier())]),
        {"clf__n_neighbors": [3, 5, 7, 11, 15, 21, 31],
         "clf__weights": ["uniform", "distance"],
         "clf__p": [1, 2]},
    ),
    "Naive Bayes (Gaussian)": (
        Pipeline([("clf", GaussianNB())]),
        {"clf__var_smoothing": stats.loguniform(1e-11, 1e-6)},
    ),
}

Sel di bawah menjalankan nested CV untuk **semua** model. Ini bagian terlama (±2–4 menit). Anda tidak perlu mengubah apa pun di sel ini.

In [13]:
DIST = {
    "Logistic Regression": {"clf__C": stats.loguniform(1e-3, 1e2)},
    "SVM (RBF)": {"clf__C": stats.loguniform(1e-2, 1e3),
                  "clf__gamma": stats.loguniform(1e-4, 1e0)},
    "Random Forest": {"clf__n_estimators": [100, 200],
                      "clf__max_depth": [None, 5, 10],
                      "clf__max_features": ["sqrt", "log2", 0.5],
                      "clf__min_samples_leaf": [1, 2, 5]},
}

def evaluasi_nested(estimator, param_dist, X_train, y_train):
    """Tuning di lipatan dalam (TUNING_BUDGET percobaan), penilaian di lipatan luar."""
    cari = RandomizedSearchCV(
        estimator, param_dist, n_iter=TUNING_BUDGET, cv=CV_DALAM,
        scoring=SCORING[METRIK_UTAMA], random_state=RANDOM_STATE, n_jobs=-1)
    return cross_validate(cari, X_train, y_train, cv=CV_LUAR, scoring=SCORING)

KANDIDAT = {}       # dataset -> {model: (estimator, distribusi)}
SKOR_FOLD = {}      # (dataset, model) -> skor metrik utama per fold luar
for nama in DATASET:
    X_train, _, y_train, _ = SPLIT[nama]
    kandidat = {m: (e, DIST[m]) for m, e in buat_model_wajib().items()}
    kandidat.update(MODEL_SAYA)
    KANDIDAT[nama] = kandidat
    for model, (est, dist) in kandidat.items():
        t0 = time.time()
        hasil = evaluasi_nested(est, dist, X_train, y_train)
        TABEL[(nama, model, "tuned")] = ringkas(hasil)
        SKOR_FOLD[(nama, model)] = hasil["test_" + METRIK_UTAMA]
        print(f"[{nama}] {model:22s} selesai dalam {time.time()-t0:5.1f} dtk")
print(f"\nModel yang dibandingkan: {list(KANDIDAT[next(iter(DATASET))])}")
print(f"Anggaran tuning per model: {TUNING_BUDGET} percobaan (sama untuk semua)")

[A (mudah)] Logistic Regression    selesai dalam   8.5 dtk
[A (mudah)] SVM (RBF)              selesai dalam   5.4 dtk
[A (mudah)] Random Forest          selesai dalam  85.1 dtk
[A (mudah)] k-NN                   selesai dalam   2.9 dtk
[A (mudah)] Naive Bayes (Gaussian) selesai dalam   2.7 dtk
[B (sulit)] Logistic Regression    selesai dalam   3.8 dtk
[B (sulit)] SVM (RBF)              selesai dalam  19.6 dtk
[B (sulit)] Random Forest          selesai dalam 177.8 dtk
[B (sulit)] k-NN                   selesai dalam   3.9 dtk
[B (sulit)] Naive Bayes (Gaussian) selesai dalam   2.2 dtk

Model yang dibandingkan: ['Logistic Regression', 'SVM (RBF)', 'Random Forest', 'k-NN', 'Naive Bayes (Gaussian)']
Anggaran tuning per model: 12 percobaan (sama untuk semua)


**Interpretasi** (1–2 kalimat, sebut angkanya): _(tulis di sini)_

## Blok 6 — Default vs tuned: seberapa besar tuning mengubah cerita?

In [ ]:
for nama in DATASET:
    print(f"\n=== Dataset {nama} ===")
    print(tampilkan(TABEL, nama).to_string(index=False))

print(f"\nSelisih {METRIK_UTAMA} (tuned - default):")
for nama in DATASET:
    for model in KANDIDAT[nama]:
        if (nama, model, "default") in TABEL:
            d = TABEL[(nama, model, "tuned")][METRIK_UTAMA][0] - TABEL[(nama, model, "default")][METRIK_UTAMA][0]
            print(f"  [{nama}] {model:22s} {d:+.4f}")
        else:
            print(f"  [{nama}] {model:22s} (tanpa versi default; model tambahan Anda)")

Bila satu model diberi tuning dan yang lain tidak, apakah perbandingannya adil? Bandingkan besar selisih di atas dengan selisih antar model.

**Interpretasi** (1–2 kalimat, sebut angkanya): _(tulis di sini)_

## Blok 7 — Apakah perbedaannya nyata? Uji berpasangan terkoreksi

Skor dari folds yang tumpang-tindih **tidak independen**, sehingga uji-t biasa terlalu percaya diri. Kita memakai **uji-t berpasangan terkoreksi** (Nadeau & Bengio, 2003), yang menambah penyesuaian pada variansi. Bandingkan model terbaik terhadap tiap model lain, dan bedakan dua pertanyaan: *signifikan secara statistik?* dan *cukup besar untuk berarti secara praktis?*

In [ ]:
AMBANG_PRAKTIS = 0.01     # selisih PR-AUC yang Anda anggap berarti; alasannya ditulis di laporan

def uji_berpasangan_terkoreksi(a, b, n_train, n_test):
    d = np.asarray(a) - np.asarray(b)
    k = len(d)
    v = d.var(ddof=1)
    if v == 0:
        return d.mean(), 1.0
    t = d.mean() / np.sqrt((1 / k + n_test / n_train) * v)
    return d.mean(), 2 * stats.t.sf(abs(t), k - 1)

for nama in DATASET:
    X_train = SPLIT[nama][0]
    n_tr = int(len(X_train) * 0.8); n_te = len(X_train) - n_tr
    rata = {m: SKOR_FOLD[(nama, m)].mean() for m in KANDIDAT[nama]}
    terbaik = max(rata, key=rata.get)
    print(f"\n=== Dataset {nama}: terbaik menurut {METRIK_UTAMA} = {terbaik} ({rata[terbaik]:.4f}) ===")
    for m in KANDIDAT[nama]:
        if m == terbaik:
            continue
        selisih, p = uji_berpasangan_terkoreksi(SKOR_FOLD[(nama, terbaik)], SKOR_FOLD[(nama, m)], n_tr, n_te)
        signif = "signifikan" if p < 0.05 else "TIDAK signifikan"
        praktis = "berarti" if abs(selisih) >= AMBANG_PRAKTIS else "di bawah ambang praktis"
        print(f"  {terbaik} vs {m:22s} selisih={selisih:+.4f}  p={p:.3f}  -> {signif}, {praktis}")

Kapan selisih itu signifikan tetapi tidak berarti, dan kapan sebaliknya? Apa yang terjadi pada Dataset A dibanding B?

**Interpretasi** (1–2 kalimat, sebut angkanya): _(tulis di sini)_

## Blok 8 — Bias–variance: kompleksitas model

Satu pohon keputusan dengan kedalaman bertambah: skor latih naik terus, skor validasi berhenti naik lalu turun. Jarak keduanya adalah variance yang Anda bayar.

In [ ]:
from sklearn.model_selection import validation_curve
from sklearn.tree import DecisionTreeClassifier

nama_kurva = "B (sulit)"
X_train, _, y_train, _ = SPLIT[nama_kurva]
kedalaman = list(range(1, 13))
skor_latih, skor_val = validation_curve(
    DecisionTreeClassifier(random_state=RANDOM_STATE), X_train, y_train,
    param_name="max_depth", param_range=kedalaman, cv=CV_DALAM,
    scoring=SCORING[METRIK_UTAMA])

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(kedalaman, skor_latih.mean(1), marker="o", label="latih")
ax.plot(kedalaman, skor_val.mean(1), marker="o", label="validasi (CV)")
ax.set_xlabel("max_depth"); ax.set_ylabel(METRIK_UTAMA)
ax.set_title(f"Kompleksitas vs generalisasi — Decision Tree, dataset {nama_kurva}")
ax.legend(); plt.tight_layout()
plt.savefig("outputs/kurva_kompleksitas_dt.png", dpi=150); plt.show()
print("Kedalaman terbaik menurut validasi:", kedalaman[int(np.argmax(skor_val.mean(1)))])

Di kedalaman berapa model mulai menghafal? Kaitkan dengan apa yang Anda lihat pada Random Forest di Blok 5 (mengapa averaging membantu?).

**Interpretasi** (1–2 kalimat, sebut angkanya): _(tulis di sini)_

## Blok 9 — Pilih model, buka test set **satu kali**

Model dipilih **hanya** dari skor nested-CV di data latih (tidak dari test set). Hyperparameter model terpilih dicari ulang pada seluruh data latih, lalu dinilai satu kali di test set. Sel ini satu-satunya tempat test set disentuh.

In [ ]:
from sklearn.metrics import (roc_auc_score, average_precision_score, f1_score,
                             accuracy_score, confusion_matrix, classification_report)

HASIL_TEST = {}
for nama in DATASET:
    X_train, X_test, y_train, y_test = SPLIT[nama]
    rata = {m: SKOR_FOLD[(nama, m)].mean() for m in KANDIDAT[nama]}
    terbaik = max(rata, key=rata.get)
    est, dist = KANDIDAT[nama][terbaik]

    cari = RandomizedSearchCV(est, dist, n_iter=TUNING_BUDGET, cv=CV_DALAM,
                              scoring=SCORING[METRIK_UTAMA], random_state=RANDOM_STATE, n_jobs=-1)
    cari.fit(X_train, y_train)

    proba = cari.predict_proba(X_test)[:, 1]
    pred = cari.predict(X_test)
    HASIL_TEST[nama] = {"model": terbaik,
                        "roc_auc": roc_auc_score(y_test, proba),
                        "pr_auc": average_precision_score(y_test, proba),
                        "f1": f1_score(y_test, pred),
                        "accuracy": accuracy_score(y_test, pred)}
    print(f"\n=== Dataset {nama}: model terpilih = {terbaik} ===")
    print("hyperparameter :", cari.best_params_)
    print({k: round(v, 3) for k, v in HASIL_TEST[nama].items() if k != "model"})
    print("acuan test     : akurasi tebakan-mayoritas =", round(max(y_test.mean(), 1 - y_test.mean()), 3),
          "| PR-AUC acak =", round(y_test.mean(), 3))
    print("confusion matrix [baris=aktual, kolom=prediksi]:\n", confusion_matrix(y_test, pred))
    print(classification_report(y_test, pred, digits=3))

Apakah skor test konsisten dengan nested-CV (Blok 5)? Bila jauh berbeda, apa dugaan Anda?

**Interpretasi** (1–2 kalimat, sebut angkanya): _(tulis di sini)_

## Blok 10 — Ringkasan lintas dataset

In [ ]:
fig, axes = plt.subplots(1, len(DATASET), figsize=(11, 4), sharey=False)
for ax, nama in zip(np.atleast_1d(axes), DATASET):
    model = list(KANDIDAT[nama])
    rata = [SKOR_FOLD[(nama, m)].mean() for m in model]
    sd = [SKOR_FOLD[(nama, m)].std() for m in model]
    ax.bar(model, rata, yerr=sd, capsize=4)
    ax.axhline(TABEL[(nama, "Dummy (mayoritas)", "-")][METRIK_UTAMA][0], color="gray", ls="--", label="Dummy")
    ax.set_title(f"Dataset {nama}"); ax.set_ylabel(f"{METRIK_UTAMA} (nested CV, rata-rata ± std)")
    ax.tick_params(axis="x", rotation=25); ax.legend()
plt.tight_layout(); plt.savefig("outputs/perbandingan_model.png", dpi=150); plt.show()

print("Peringkat model menurut", METRIK_UTAMA, "(nested CV):")
for nama in DATASET:
    urut = sorted(KANDIDAT[nama], key=lambda m: -SKOR_FOLD[(nama, m)].mean())
    print(f"  {nama}: " + " > ".join(urut))

Apakah peringkat model sama di kedua dataset? Apa artinya bagi klaim bahwa satu model adalah "strong baseline" yang berlaku umum?

**Interpretasi** (1–2 kalimat, sebut angkanya): _(tulis di sini)_

## Blok 11 — Log reproduksibilitas

In [ ]:
print("scikit-learn      :", sklearn.__version__)
print("RANDOM_STATE      :", RANDOM_STATE)
print("TUNING_BUDGET     :", TUNING_BUDGET)
print("CV luar / dalam   : 5x2 berulang / 3 lipatan")
print("Metrik utama      :", METRIK_UTAMA)
print("Model tambahan    :", list(MODEL_SAYA) or "BELUM DIISI")
print(f"Waktu total       : {(time.time()-T_MULAI)/60:.1f} menit")

## Opsional — pakai protokol ini pada dataset penelitian Anda

Tidak dinilai otomatis, tetapi inilah gunanya praktikum ini. Buka komentar, isi `X_riset` dan `y_riset` (ingat: split dan simpan test set **sebelum** apa pun), lalu pakai `evaluasi_nested` yang sama.

```python
# X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(X_riset, y_riset, test_size=0.2, stratify=y_riset, random_state=RANDOM_STATE)
# for model, (est, dist) in KANDIDAT["A (mudah)"].items():
#     print(model, ringkas(evaluasi_nested(est, dist, X_train_r, y_train_r))[METRIK_UTAMA])
```

## Selesai — langkah berikutnya

1. Pastikan semua sel sudah dijalankan (**Runtime → Run all**), lalu simpan ke GitHub.
2. Isi `laporan/LAPORAN.md`: empat temuan, Kartu Baseline Penelitian versi akhir, audit klaim, refleksi.
3. Push. Buka tab **Actions** dan pastikan pemeriksaan berwarna hijau.